# Target samples

Lists of known astronomical objects for inspecting the OVRO-LWA metacatalog in
[`metacatalog_query.ipynb`](metacatalog_query.ipynb). Paste a row's `coord` string
(decimal degrees, `RA DEC`) into that notebook's nearest-source box.

This notebook does **not** load the metacatalog. Each section downloads (or reuses a
cached copy of) a published catalog and shows it in a Panel **Tabulator** (sortable
columns, per-column header filters). Paste a row's `coord` string into
`metacatalog_query.ipynb`.

| Section | Sample | Source |
| ------- | ------ | ------ |
| Local Galaxies | NED-LVS galaxies, z ≤ 0.025, nearest first | [NED-LVS](https://ned.ipac.caltech.edu/NED::LVS/) (Cook et al. 2023); CLU lineage |
| Galaxy Clusters | MCXC X-ray clusters, nearest first | [Piffaretti et al. 2011](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/534/A109) |
| AGN Jets | Nature record-holders + LoTSS GRGs | [Oei et al. 2024](https://scixplorer.org/abs/2024Natur.633..537O/abstract); [Dabhade et al. 2020](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/635/A5) |
| Supernova Remnants | Green Galactic SNR catalogue | [Green 2024 Oct / 2025](https://www.mrao.cam.ac.uk/surveys/snrs/) |
| Pulsars | ATNF Pulsar Catalogue | [Manchester et al. 2005](https://www.atnf.csiro.au/research/pulsar/psrcat/) |
| X-ray/optical systems | Rodriguez (2024) Table 2 | [arXiv:2401.09537](https://arxiv.org/html/2401.09537v1#A2) |

**Run cells in order.**


In [1]:
from __future__ import annotations

import re
import tarfile
import urllib.request
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
import panel as pn
from astropy.coordinates import BarycentricMeanEcliptic, SkyCoord
from astropy.io import ascii

from lwa_catalog.analyze import load_nedlvs_catalog
from lwa_catalog.constants import NEDLVS_DEFAULT_PATH, REFERENCE_CATALOGS_DIR

pn.extension("tabulator")

# --- operator config -------------------------------------------------------
CACHE_DIR = Path(REFERENCE_CATALOGS_DIR)
NEDLVS_PATH = NEDLVS_DEFAULT_PATH
MAX_REDSHIFT_GALAXIES = 0.025  # ~100 Mpc
MIN_DIST_MPC = 0.05  # 50 kpc; drops Galactic-star contaminants with z ~ 0
TABLE_HEIGHT = 420
TABLE_PAGE_SIZE = 25
TABLE_REMOTE_THRESHOLD = 5000  # remote pagination above this many rows
USER_AGENT = "lwa-catalog/target_samples (claw@astro.caltech.edu)"

CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_url(url: str, dest: Path, *, force: bool = False) -> Path:
    '''Download *url* to *dest* unless the file already exists.'''
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0 and not force:
        print(f"cached {dest} ({dest.stat().st_size:,} bytes)")
        return dest
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as resp:
        dest.write_bytes(resp.read())
    print(f"wrote {dest} ({dest.stat().st_size:,} bytes)")
    return dest


def read_cds(data_path: Path, readme_path: Path):
    '''Read a CDS/VizieR fixed-width table using its ReadMe.'''
    return ascii.read(str(data_path), readme=str(readme_path))


def add_coord(df: pd.DataFrame) -> pd.DataFrame:
    '''Add a `coord` column (`RA DEC` decimal degrees) for metacatalog_query.'''
    out = df.copy()
    out["coord"] = [
        f"{float(ra):.6f} {float(dec):.6f}" for ra, dec in zip(out["RA"], out["DEC"], strict=True)
    ]
    return out


def show_table(df: pd.DataFrame, *, height: int | None = None) -> pn.widgets.Tabulator:
    '''Panel Tabulator with sortable columns and per-column header filters.'''
    print(f"{len(df)} rows, columns: {list(df.columns)}")
    n = len(df)
    if n <= 12:
        pagination = None
        page_size = n
        height = height or min(TABLE_HEIGHT, 80 + 28 * max(n, 1))
    elif n > TABLE_REMOTE_THRESHOLD:
        pagination = "remote"
        page_size = TABLE_PAGE_SIZE
        height = height or TABLE_HEIGHT
    else:
        pagination = "local"
        page_size = TABLE_PAGE_SIZE
        height = height or TABLE_HEIGHT
    return pn.widgets.Tabulator(
        df,
        pagination=pagination,
        page_size=page_size,
        height=height,
        sizing_mode="stretch_width",
        layout="fit_data_table",
        header_filters=True,
        show_index=False,
        disabled=True,
        sortable=True,
        selectable=1,
    )


print("CACHE_DIR =", CACHE_DIR.resolve())
print("NEDLVS_PATH =", Path(NEDLVS_PATH).resolve(), "(exists)" if Path(NEDLVS_PATH).is_file() else "(missing)")
print("MAX_REDSHIFT_GALAXIES =", MAX_REDSHIFT_GALAXIES)
print("MIN_DIST_MPC =", MIN_DIST_MPC)


CACHE_DIR = /fast/claw/catalogs
NEDLVS_PATH = /fast/claw/catalogs/NEDLVS_current.fits (exists)
MAX_REDSHIFT_GALAXIES = 0.025
MIN_DIST_MPC = 0.05


## Local Galaxies

The [Census of the Local Universe](https://ui.adsabs.harvard.edu/abs/2019ApJ...880....7C)
(CLU; Cook et al. 2019) is a nearby-galaxy compilation for gravitational-wave
follow-up. This project already holds the all-sky successor sample, the
[NED Local Volume Sample](https://ned.ipac.caltech.edu/NED::LVS/) (NED-LVS;
Cook et al. 2023), at `NEDLVS_PATH`. NED-LVS includes the CLU galaxies and
extends to D ~ 1000 Mpc.

Keep `objtype == "G"` with catalog redshift 0 ≤ z ≤ `MAX_REDSHIFT_GALAXIES`
(default 0.025, ~100 Mpc) and `DistMpc >= MIN_DIST_MPC` (default 0.05 Mpc = 50 kpc,
so the Magellanic Clouds remain and Galactic stars with z ~ 0 are dropped). Sort by
NED-LVS `DistMpc` (redshift-independent distances preferred below 200 Mpc).

The full sorted table is `local_galaxies`. Paste `coord` into `metacatalog_query.ipynb`.


In [2]:
ned = load_nedlvs_catalog(NEDLVS_PATH)
z = pd.to_numeric(ned["z"], errors="coerce")
dist = pd.to_numeric(ned["DistMpc"], errors="coerce")
is_galaxy = ned["objtype"].astype(str).str.strip().eq("G")
names = ned["objname"].map(
    lambda x: x.decode("utf-8", errors="replace") if isinstance(x, (bytes, bytearray)) else str(x)
)
local_galaxies = (
    ned.loc[is_galaxy & np.isfinite(z.to_numpy()) & (z >= 0.0) & (z <= MAX_REDSHIFT_GALAXIES)]
    .assign(name=names, DistMpc=dist, z=z)
)
local_galaxies = local_galaxies.loc[local_galaxies["DistMpc"] >= MIN_DIST_MPC].copy()
local_galaxies = local_galaxies.sort_values(
    "DistMpc", na_position="last", kind="mergesort"
).reset_index(drop=True)
local_galaxies = add_coord(local_galaxies)
local_galaxies = local_galaxies[
    ["name", "RA", "DEC", "coord", "z", "DistMpc", "Diam_arcsec", "Mstar", "SFR_hybrid", "SFR_W4"]
]

print(
    f"NED-LVS galaxies with z <= {MAX_REDSHIFT_GALAXIES}: {len(local_galaxies):,}"
)
print(
    "DistMpc: min={:.3f}  median={:.1f}  max={:.1f}".format(
        local_galaxies["DistMpc"].min(),
        local_galaxies["DistMpc"].median(),
        local_galaxies["DistMpc"].max(),
    )
)
show_table(local_galaxies)


NED-LVS galaxies with z <= 0.025: 88,891
DistMpc: min=0.050  median=74.9  max=944.5
88891 rows, columns: ['name', 'RA', 'DEC', 'coord', 'z', 'DistMpc', 'Diam_arcsec', 'Mstar', 'SFR_hybrid', 'SFR_W4']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='remote', show_index=False, sizing_mode='stretch_width', value=              ...)

## Galaxy Clusters

X-ray clusters from the [MCXC meta-catalogue](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/534/A109)
(Piffaretti et al. 2011; VizieR `J/A+A/534/A109`). MCXC homogenises ROSAT All-Sky
Survey and serendipitous cluster catalogues to Δ=500 masses and radii.

Sorted by redshift (nearest first). The full table is `galaxy_clusters`.


In [3]:
mcxc_dir = CACHE_DIR / "mcxc"
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/534/A109/ReadMe", mcxc_dir / "ReadMe")
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/534/A109/mcxc.dat", mcxc_dir / "mcxc.dat")
mcxc = read_cds(mcxc_dir / "mcxc.dat", mcxc_dir / "ReadMe").to_pandas()

aname = mcxc["AName"].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA})
oname = mcxc["OName"].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA})
galaxy_clusters = pd.DataFrame(
    {
        "name": aname.fillna(oname).fillna(mcxc["MCXC"].astype(str)),
        "MCXC": mcxc["MCXC"].astype(str),
        "OName": mcxc["OName"].astype(str).str.strip(),
        "RA": pd.to_numeric(mcxc["RAdeg"], errors="coerce"),
        "DEC": pd.to_numeric(mcxc["DEdeg"], errors="coerce"),
        "z": pd.to_numeric(mcxc["z"], errors="coerce"),
        "M500": pd.to_numeric(mcxc["M500"], errors="coerce"),
        "R500_Mpc": pd.to_numeric(mcxc["R500"], errors="coerce"),
        "L500": pd.to_numeric(mcxc["L500"], errors="coerce"),
        "Cat": mcxc["Cat"].astype(str).str.strip(),
    }
)
galaxy_clusters = (
    galaxy_clusters.dropna(subset=["RA", "DEC"])
    .sort_values("z", na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
galaxy_clusters = add_coord(galaxy_clusters)
galaxy_clusters = galaxy_clusters[
    ["name", "MCXC", "OName", "RA", "DEC", "coord", "z", "M500", "R500_Mpc", "L500", "Cat"]
]
show_table(galaxy_clusters)


cached /fast/claw/catalogs/mcxc/ReadMe (7,064 bytes)
cached /fast/claw/catalogs/mcxc/mcxc.dat (415,572 bytes)
1743 rows, columns: ['name', 'MCXC', 'OName', 'RA', 'DEC', 'coord', 'z', 'M500', 'R500_Mpc', 'L500', 'Cat']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=                          ...)

## AGN Jets

Giant radio galaxies (Mpc-scale AGN jets) cited in
[Oei et al. 2024, *Nature* 633, 537](https://scixplorer.org/abs/2024Natur.633..537O/abstract)
([arXiv:2411.08630](https://arxiv.org/abs/2411.08630)):

- **Porphyrion** (the 7 Mpc discovery; host J152932.16+601534.4)
- Previous record-length outflows named in that paper: **Alcyoneus** (Oei et al. 2022),
  **J1420−0545** (Machalski et al. 2008), and **3C 236**

plus the systematic LoTSS DR1 GRG catalogue of
[Dabhade et al. 2020](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/635/A5)
(VizieR `J/A+A/635/A5`; 239 sources), which is the public catalog from that
paper's LoTSS GRG series.

`agn_jets_named` is the Nature record-holder list (largest first).
`agn_jets` is Dabhade+2020 sorted by projected linear size.


In [4]:
named_rows = [
    {
        "name": "Porphyrion",
        "host": "J152932.16+601534.4",
        "sky": SkyCoord("15h29m32.16s", "+60d15m34.4s", frame="icrs"),
        "z": 0.896,
        "lp_Mpc": 6.43,
        "ref": "Oei+2024 Nature",
    },
    {
        "name": "Alcyoneus",
        "host": "SDSS J081421.68+522410.0",
        "sky": SkyCoord(ra=123.590372 * u.deg, dec=52.402795 * u.deg, frame="icrs"),
        "z": 0.24674,
        "lp_Mpc": 5.00,
        "ref": "Oei+2022 A&A",
    },
    {
        "name": "J1420-0545",
        "host": "J1420-0545",
        "sky": SkyCoord("14h20m23.8s", "-05d45m28.8s", frame="icrs"),
        "z": 0.3067,
        "lp_Mpc": 4.9,
        "ref": "Machalski+2008",
    },
    {
        "name": "3C 236",
        "host": "3C 236",
        "sky": SkyCoord("10h06m01.735s", "+34d54m10.43s", frame="icrs"),
        "z": 0.099358,
        "lp_Mpc": 4.6,
        "ref": "Willis+1974 / Schoenmakers+2000",
    },
]
agn_jets_named = pd.DataFrame(
    {
        "name": [r["name"] for r in named_rows],
        "host": [r["host"] for r in named_rows],
        "RA": [float(r["sky"].ra.deg) for r in named_rows],
        "DEC": [float(r["sky"].dec.deg) for r in named_rows],
        "z": [r["z"] for r in named_rows],
        "lp_Mpc": [r["lp_Mpc"] for r in named_rows],
        "ref": [r["ref"] for r in named_rows],
    }
)
agn_jets_named = add_coord(agn_jets_named)
print("Nature-paper record holders (projected length lp):")
named_table = show_table(agn_jets_named)

grg_dir = CACHE_DIR / "dabhade_grg"
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/635/A5/ReadMe", grg_dir / "ReadMe")
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/635/A5/tablea1.dat",
    grg_dir / "tablea1.dat",
)
grg = read_cds(grg_dir / "tablea1.dat", grg_dir / "ReadMe")
sign = np.where(np.asarray(grg["DE-"], dtype=str) == "-", -1.0, 1.0)
ra = (np.asarray(grg["RAh"], dtype=float) + np.asarray(grg["RAm"], dtype=float) / 60.0
      + np.asarray(grg["RAs"], dtype=float) / 3600.0) * 15.0
dec = sign * (
    np.asarray(grg["DEd"], dtype=float)
    + np.asarray(grg["DEm"], dtype=float) / 60.0
    + np.asarray(grg["DEs"], dtype=float) / 3600.0
)
names = []
for i in range(len(grg)):
    c = SkyCoord(ra=ra[i] * u.deg, dec=dec[i] * u.deg, frame="icrs")
    ra_h = c.ra.to_string(unit=u.hourangle, sep="", precision=1, pad=True)
    dec_d = c.dec.to_string(sep="", precision=1, alwayssign=True, pad=True)
    names.append(f"J{ra_h}{dec_d}")

agn_jets = pd.DataFrame(
    {
        "name": names,
        "RA": ra,
        "DEC": dec,
        "host_class": np.asarray(grg["Class"], dtype=str),
        "z": np.asarray(grg["z"], dtype=float),
        "Size_arcsec": np.asarray(grg["Size"], dtype=float),
        "SizeP_Mpc": np.asarray(grg["SizeP"], dtype=float),
        "S144_mJy": np.asarray(grg["S144MHz"], dtype=float),
        "FR": np.asarray(grg["FR"], dtype=str),
    }
)
agn_jets = (
    agn_jets.sort_values("SizeP_Mpc", ascending=False, na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
agn_jets = add_coord(agn_jets)
print("Dabhade+2020 LoTSS GRGs (largest projected size first):")
pn.Column(named_table, show_table(agn_jets))


Nature-paper record holders (projected length lp):
4 rows, columns: ['name', 'host', 'RA', 'DEC', 'z', 'lp_Mpc', 'ref', 'coord']
cached /fast/claw/catalogs/dabhade_grg/ReadMe (8,975 bytes)
cached /fast/claw/catalogs/dabhade_grg/tablea1.dat (27,524 bytes)
Dabhade+2020 LoTSS GRGs (largest projected size first):
239 rows, columns: ['name', 'RA', 'DEC', 'host_class', 'z', 'Size_arcsec', 'SizeP_Mpc', 'S144_mJy', 'FR', 'coord']


Column
    [0] Tabulator(disabled=True, header_filters=True, height=192, page_size=4, show_index=False, sizing_mode='stretch_width', value=         name  ...)
    [1] Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=              ...)

## Supernova Remnants

[Green's Catalogue of Galactic Supernova Remnants](https://www.mrao.cam.ac.uk/surveys/snrs/)
(2024 October version; 310 remnants). Machine-readable summary from VizieR
[`VII/297`](https://cdsarc.cds.unistra.fr/viz-bin/cat/VII/297)
(Green 2025, JApA, 46, 14).

Sorted by 1 GHz flux density (brightest first). The full table is `supernova_remnants`.
Cite Green (2025) and the 2024 October web catalogue if you use these positions.


In [5]:
snr_dir = CACHE_DIR / "green_snr"
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/VII/297/ReadMe", snr_dir / "ReadMe")
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/VII/297/snrs.dat", snr_dir / "snrs.dat")
snr = read_cds(snr_dir / "snrs.dat", snr_dir / "ReadMe")
sign = np.where(np.asarray(snr["DE-"], dtype=str) == "-", -1.0, 1.0)
ra = (np.asarray(snr["RAh"], dtype=float) + np.asarray(snr["RAm"], dtype=float) / 60.0
      + np.asarray(snr["RAs"], dtype=float) / 3600.0) * 15.0
dec = sign * (
    np.asarray(snr["DEd"], dtype=float) + np.asarray(snr["DEm"], dtype=float) / 60.0
)
other = pd.Series(np.asarray(snr["Names"], dtype=str)).str.strip()
supernova_remnants = pd.DataFrame(
    {
        "name": np.asarray(snr["SNR"], dtype=str),
        "other_names": other,
        "RA": ra,
        "DEC": dec,
        "type": pd.Series(np.asarray(snr["type"], dtype=str)).str.strip(),
        "MajDiam_arcmin": np.asarray(snr["MajDiam"], dtype=float),
        "MinDiam_arcmin": np.asarray(snr["MinDiam"], dtype=float),
        "S_1GHz_Jy": np.asarray(snr["S(1GHz)"], dtype=float),
        "sp_index": np.asarray(snr["Sp-Index"], dtype=float),
    }
)
supernova_remnants = (
    supernova_remnants.sort_values(
        "S_1GHz_Jy", ascending=False, na_position="last", kind="mergesort"
    )
    .reset_index(drop=True)
)
supernova_remnants = add_coord(supernova_remnants)
show_table(supernova_remnants)


cached /fast/claw/catalogs/green_snr/ReadMe (9,414 bytes)
cached /fast/claw/catalogs/green_snr/snrs.dat (19,799 bytes)
310 rows, columns: ['name', 'other_names', 'RA', 'DEC', 'type', 'MajDiam_arcmin', 'MinDiam_arcmin', 'S_1GHz_Jy', 'sp_index', 'coord']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=            name          ...)

## Pulsars

The [ATNF Pulsar Catalogue](https://www.atnf.csiro.au/research/pulsar/psrcat/)
(Manchester et al. 2005; live `psrcat.db` from the current public package).
Positions are ICRS: `RAJ`/`DECJ` when present, otherwise ecliptic
`ELONG`/`ELAT` converted to ICRS.

Sorted by YMW16 DM-distance (`DIST_DM`, kpc), nearest first. The full table is
`pulsars`. Cite Manchester et al. (2005) and the ATNF web catalogue.


In [6]:
def _psrcat_first_value(block: str) -> dict[str, str]:
    rec: dict[str, str] = {}
    for line in block.splitlines():
        if not line or line.startswith("#"):
            continue
        match = re.match(r"^([A-Z0-9_]+)\s+(\S+)", line)
        if match is None:
            continue
        rec.setdefault(match.group(1), match.group(2))
    return rec


def load_atnf_psrcat(db_path: Path) -> pd.DataFrame:
    text = db_path.read_text(encoding="latin1")
    rows: list[dict[str, object]] = []
    for block in re.split(r"\n@.*\n", text):
        rec = _psrcat_first_value(block)
        name = rec.get("PSRJ") or rec.get("PSRB")
        if not name:
            continue
        ra = dec = np.nan
        if "RAJ" in rec and "DECJ" in rec:
            sky = SkyCoord(rec["RAJ"], rec["DECJ"], unit=(u.hourangle, u.deg), frame="icrs")
            ra, dec = float(sky.ra.deg), float(sky.dec.deg)
        elif "ELONG" in rec and "ELAT" in rec:
            sky = SkyCoord(
                lon=float(rec["ELONG"]) * u.deg,
                lat=float(rec["ELAT"]) * u.deg,
                frame=BarycentricMeanEcliptic(),
            ).icrs
            ra, dec = float(sky.ra.deg), float(sky.dec.deg)
        else:
            continue
        p0 = rec.get("P0")
        if p0 is None and "F0" in rec:
            f0 = float(rec["F0"])
            p0 = f"{1.0 / f0:.12g}" if f0 else None
        dist = rec.get("DIST") or rec.get("DIST_DM")
        rows.append(
            {
                "name": name,
                "RA": ra,
                "DEC": dec,
                "P0_s": pd.to_numeric(p0, errors="coerce"),
                "DM": pd.to_numeric(rec.get("DM"), errors="coerce"),
                "S400_mJy": pd.to_numeric(rec.get("S400"), errors="coerce"),
                "S1400_mJy": pd.to_numeric(rec.get("S1400"), errors="coerce"),
                "Dist_kpc": pd.to_numeric(dist, errors="coerce"),
                "Type": rec.get("TYPE", ""),
                "Assoc": rec.get("ASSOC", ""),
            }
        )
    return pd.DataFrame(rows)


psrcat_dir = CACHE_DIR / "psrcat"
pkg = cache_url(
    "https://www.atnf.csiro.au/research/pulsar/psrcat/downloads/psrcat_pkg.tar.gz",
    psrcat_dir / "psrcat_pkg.tar.gz",
)
db_path = psrcat_dir / "psrcat.db"
if not db_path.is_file() or db_path.stat().st_size == 0:
    with tarfile.open(pkg, mode="r:gz") as tf:
        member = next(m for m in tf.getmembers() if m.name.endswith("psrcat.db"))
        extracted = tf.extractfile(member)
        if extracted is None:
            raise FileNotFoundError("psrcat.db missing from ATNF tarball")
        db_path.write_bytes(extracted.read())
    print(f"extracted {db_path} ({db_path.stat().st_size:,} bytes)")
else:
    print(f"cached {db_path} ({db_path.stat().st_size:,} bytes)")

header = db_path.read_text(encoding="latin1", errors="replace").splitlines()[0]
print("ATNF", header.lstrip("#").strip())

pulsars = load_atnf_psrcat(db_path)
pulsars = pulsars.sort_values("Dist_kpc", na_position="last", kind="mergesort").reset_index(drop=True)
pulsars = add_coord(pulsars)
show_table(pulsars)


cached /fast/claw/catalogs/psrcat/psrcat_pkg.tar.gz (1,224,627 bytes)
cached /fast/claw/catalogs/psrcat/psrcat.db (10,432,859 bytes)
ATNF CATALOGUE 2.8.1
4393 rows, columns: ['name', 'RA', 'DEC', 'P0_s', 'DM', 'S400_mJy', 'S1400_mJy', 'Dist_kpc', 'Type', 'Assoc', 'coord']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=            name          ...)

## X-ray/optical systems

Galactic compact-object and related binaries from Table 2 of
[Rodriguez 2024, arXiv:2401.09537](https://arxiv.org/html/2401.09537v1#A2)
(*From Active Stars to Black Holes*). These are the literature systems plotted
on the X-ray Main Sequence (redbacks, black widows, LMXBs, HMXBs, symbiotic
stars, supersoft sources).

Gaia DR3 ICRS positions are included with each row (resolved once from ESA TAP).
The table is `xray_optical`, in the paper's class order.


In [7]:
# Rodriguez 2024 Table 2, with Gaia DR3 ICRS positions (queried once from ESA TAP).
# xray_ref: (1) spider binaries; (2) BH LMXBs; (3) NS LMXBs; (4) HMXBs;
# (5) symbiotic NS; (6) symbiotic WD; (7) 4XMM-DR13 SSS.
XRAY_OPTICAL_ROWS = [
    ("J0212+5320", "Redback", 455282205716288384, 33.043636, 53.360781, 1),
    ("J1048+2339", "Redback", 3990037124929068032, 162.180900, 23.664831, 1),
    ("J1306-40", "Redback", 6140785016794586752, 196.734467, -40.589833, 1),
    ("J1431-4715", "Redback", 6098156298150016768, 217.935887, -47.257676, 1),
    ("J1622-0315", "Redback", 4358428942492430336, 245.748445, -3.260352, 1),
    ("J1628-3205", "Redback", 6025344817107454464, 247.029176, -32.096950, 1),
    ("J1723-2837", "Redback", 4059795674516044800, 260.846581, -28.632665, 1),
    ("J1803-6707", "Redback", 6436867623955512064, 270.767647, -67.126710, 1),
    ("J1816+4510", "Redback", 2115337192179377792, 274.149726, 45.176069, 1),
    ("J1908+2105", "Redback", 4519819661567533696, 287.238717, 21.083920, 1),
    ("J1910-5320", "Redback", 6644467032871428992, 287.704669, -53.349200, 1),
    ("J2039-5618", "Redback", 6469722508861870080, 309.895702, -56.285910, 1),
    ("J2129-0429", "Redback", 2672030065446134656, 322.437749, -4.485225, 1),
    ("J2215+5135", "Redback", 2001168543319218048, 333.886196, 51.593455, 1),
    ("J2339-0533", "Redback", 2440660623886405504, 354.911440, -5.551465, 1),
    ("J1311-3430", "Black Widow", 6179115508262195200, 197.940506, -34.508438, 1),
    ("J1653-0158", "Black Widow", 4379227476242700928, 253.408555, -1.976915, 1),
    ("J1810+1744", "Black Widow", 4526229058440076288, 272.655365, 17.743711, 1),
    ("B1957+20", "Black Widow", 1823773960079216896, 299.903093, 20.804020, 1),
    ("GROJ0422+32", "LMXB (BH)", 172650748928103552, 65.428011, 32.907483, 2),
    ("A0620-00", "LMXB (BH)", 3118721026600835328, 95.685591, -0.345659, 2),
    ("V404 Cyg", "LMXB (BH)", 2056188624872569088, 306.015909, 33.867177, 2),
    ("XTE J1118+480", "LMXB (BH)", 789430249033567744, 169.544851, 48.036724, 2),
    ("GROJ1655-40", "LMXB (BH)", 5969790961312131456, 253.500568, -39.845800, 2),
    ("4U 2129+47", "LMXB (NS)", 1978241050130301312, 322.859207, 47.290123, 3),
    ("Cen X-4", "LMXB (NS)", 6205715168442046592, 224.591399, -31.669002, 3),
    ("Aql X-1", "LMXB (NS)", 4264296556603631872, 287.816897, 0.584941, 3),
    ("SAX J1808.4-3658", "LMXB (NS)", 4037867740522984832, 272.114284, -36.977908, 3),
    ("A0535+26", "HMXB (NS)", 3441207615229815040, 84.727392, 26.315775, 4),
    ("KS 1947+300", "HMXB (NS)", 2031939548802102656, 297.397839, 30.208808, 4),
    ("V4641 Sgr", "HMXB (BH)", 4053096388919082368, 274.840139, -25.407179, 4),
    ("Cyg X-1", "HMXB (BH)", 2059383668236814720, 299.590295, 35.201579, 4),
    ("GX 1+4", "Symbiotic (NS)", 4110236324513030656, 263.008955, -24.745600, 5),
    ("4U 1954+319", "Symbiotic (NS)", 2034031438383765760, 298.926398, 32.096930, 5),
    ("CXOGBS J173620.2-293338", "Symbiotic (NS)", 4060066227422719872, 264.084127, -29.560827, 5),
    ("4U 1700+24", "Symbiotic (NS)", 4571810378118789760, 256.643761, 23.971820, 5),
    ("NQ Gem", "Symbiotic (WD)", 868424696282795392, 112.977122, 24.503471, 6),
    ("UV Aur", "Symbiotic (WD)", 180919213811383680, 80.453799, 32.511146, 6),
    ("ZZ CMi", "Symbiotic (WD)", 3155368612444708096, 111.058322, 8.897700, 6),
    ("ER Del", "Symbiotic (WD)", 1750795043999682304, 310.693762, 8.687115, 6),
    ("CD -283719", "Symbiotic (WD)", 5608089951177429120, 105.288142, -29.106940, 6),
    ("RX J0019.8+2156", "SSS", 2800287654443977344, 4.958110, 21.947799, 7),
    ("RX J0925.7-4758", "SSS", 5422337322910734080, 141.441648, -47.971474, 7),
    ("RR Tel", "SSS", 6448785024330499456, 301.077269, -55.725891, 7),
]
xray_optical = pd.DataFrame(
    XRAY_OPTICAL_ROWS,
    columns=["name", "class", "gaia_source_id", "RA", "DEC", "xray_ref"],
)
xray_optical = add_coord(xray_optical)
xray_optical = xray_optical[
    ["name", "class", "gaia_source_id", "RA", "DEC", "coord", "xray_ref"]
]
print(f"{len(xray_optical)} systems; class counts:")
print(xray_optical["class"].value_counts().to_string())
show_table(xray_optical)


44 systems; class counts:
class
Redback           15
LMXB (BH)          5
Symbiotic (WD)     5
LMXB (NS)          4
Black Widow        4
Symbiotic (NS)     4
SSS                3
HMXB (NS)          2
HMXB (BH)          2
44 rows, columns: ['name', 'class', 'gaia_source_id', 'RA', 'DEC', 'coord', 'xray_ref']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=              ...)

## Sample sizes

Kernel variables for later cells or for copying into `metacatalog_query.ipynb`:
`local_galaxies`, `galaxy_clusters`, `agn_jets_named`, `agn_jets`,
`supernova_remnants`, `pulsars`, `xray_optical`.


In [8]:
summary = pd.DataFrame(
    [
        {"sample": "local_galaxies", "n": len(local_galaxies), "sort": "DistMpc", "note": f"NED-LVS G, z<={MAX_REDSHIFT_GALAXIES}, D>={MIN_DIST_MPC} Mpc"},
        {"sample": "galaxy_clusters", "n": len(galaxy_clusters), "sort": "z", "note": "MCXC"},
        {"sample": "agn_jets_named", "n": len(agn_jets_named), "sort": "lp_Mpc", "note": "Oei+2024 record holders"},
        {"sample": "agn_jets", "n": len(agn_jets), "sort": "SizeP_Mpc", "note": "Dabhade+2020 LoTSS GRGs"},
        {"sample": "supernova_remnants", "n": len(supernova_remnants), "sort": "S_1GHz_Jy", "note": "Green 2024 Oct"},
        {"sample": "pulsars", "n": len(pulsars), "sort": "Dist_kpc", "note": "ATNF psrcat"},
        {"sample": "xray_optical", "n": len(xray_optical), "sort": "paper order", "note": "Rodriguez 2024 Table 2"},
    ]
)
show_table(summary)


7 rows, columns: ['sample', 'n', 'sort', 'note']


Tabulator(disabled=True, header_filters=True, height=276, page_size=7, show_index=False, sizing_mode='stretch_width', value=               sample     ...)